# CodeAlpha Internship Task 3: Sales Prediction using Python
**Author**: Murali  
**Objective**: Predict sales revenue driven by advertising spend (`TV`, `Radio`, `Newspaper`), analyze channel correlation, build Multi-Variable Linear Regression and Random Forest models, and visualize actual vs. predicted sales revenue.

## 1. Import Libraries & Setup Environment

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Load & Clean Dataset

In [ ]:
df = pd.read_csv('Advertising.csv')
unnamed_cols = [c for c in df.columns if 'Unnamed' in c or 'unnamed' in c]
if unnamed_cols:
    df = df.drop(columns=unnamed_cols)

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df = df.drop_duplicates().copy()

display(df.head())
print('Missing Values:\n', df.isnull().sum())

## 3. Correlation Matrix Analysis (Identify Key Channels)

In [ ]:
corr = df.corr()
print('Correlation with Sales:\n', corr['sales'].sort_values(ascending=False))

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='viridis', fmt='.3f', linewidths=0.8)
plt.title('Correlation Matrix: Advertising Spend vs Sales Revenue')
plt.savefig(os.path.join(OUTPUT_DIR, 'sales_correlation_heatmap.png'), dpi=300)
plt.show()

## 4. Train / Test Split & Model Building

In [ ]:
X = df[['tv', 'radio', 'newspaper']]
y = df['sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Multi-Variable Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42)
}

best_r2 = -float('inf')
best_name = None
best_model = None
best_preds = None

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    print(f"--- {name} ---")
    print(f"R² Score: {r2:.4f} | MAE: {mae:.4f} | RMSE: {rmse:.4f}
")
    
    if r2 > best_r2:
        best_r2 = r2
        best_name = name
        best_model = model
        best_preds = preds

joblib.dump(best_model, os.path.join(OUTPUT_DIR, 'sales_prediction_model.pkl'))
print(f"Saved Best Model ({best_name}) binary to outputs/sales_prediction_model.pkl")

## 5. Visualizations: Feature Importance & Predictions

In [ ]:
plt.figure(figsize=(8, 5))
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values()
else:
    importances = pd.Series(np.abs(best_model.coef_), index=X_train.columns).sort_values()
importances.plot(kind='barh', color='#5cb85c')
plt.title(f'Feature Importance / Channel Impact ({best_name})')
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance_sales.png'), dpi=300)
plt.show()

plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_preds, alpha=0.85, color='#0275d8', edgecolors='k', s=60)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Actual vs Predicted Sales Revenue')
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.savefig(os.path.join(OUTPUT_DIR, 'actual_vs_predicted_sales.png'), dpi=300)
plt.show()